# PlayMind | 03. Aprendizaje no supervisado: estilos de jugador
**Proyecto final - Fundamentos de Inteligencia Artificial**  
**Entrega:** lunes 23 de noviembre de 2026 · **Equipo:** Karen Cruces, Romina Garcia, Said Jimenez, Diego Flores

**Tarea:** identificar estilos de jugador a partir de horas jugadas, sesiones por semana, nivel y compras, sin usar etiquetas.  
**Algoritmos:** K-Means y DBSCAN; PCA para visualizar.  
**Preparación:** variables escaladas con `StandardScaler`.  
**Métricas:** método del codo (inercia) y coeficiente de silueta.  
**Producto:** estilos nombrados con contenido propuesto para cada uno y modelo guardado en `models/`.

## 1. Preparación: rutas y librerías

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")
SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

# Funciona si Jupyter se abrió en la raíz del repo o dentro de notebooks/.
actual = Path.cwd().resolve()
RAIZ = actual.parent if actual.name.lower() == 'notebooks' else actual
DATOS = RAIZ / 'data' / 'raw'
PROCESADOS = RAIZ / 'data' / 'processed'
FIGURAS = RAIZ / 'reports' / 'figuras'
MODELOS = RAIZ / 'models'
for carpeta in (PROCESADOS, FIGURAS, MODELOS):
    carpeta.mkdir(parents=True, exist_ok=True)
print('Proyecto:', RAIZ)

## 2. Cargar datos tabulares
Dataset: *Predict Online Gaming Behavior Dataset* (Kaggle), ubicado en `data/raw/`.

In [ ]:
csvs = list(DATOS.rglob('*.csv')) if DATOS.exists() else []
candidatos = [p for p in csvs if any(s in p.name.lower() for s in ['gaming', 'player', 'behavior']) and 'fer' not in p.name.lower()]
if not candidatos:
    raise FileNotFoundError('No se encontró el CSV tabular en data/raw/. Archivos CSV vistos: ' + str([p.name for p in csvs]))
ruta_csv = sorted(candidatos)[0]
try:
    datos = pd.read_csv(ruta_csv)
except UnicodeDecodeError:
    datos = pd.read_csv(ruta_csv, encoding='latin1')
print('Archivo cargado:', ruta_csv.relative_to(RAIZ))
print('Filas:', datos.shape[0], '| Columnas:', datos.shape[1])
display(datos.head())

## 3. Selección y escalado de variables
Se agrupa con las cuatro variables que pide el documento: horas jugadas, sesiones por semana, nivel del jugador y compras dentro del juego. **No se usa `EngagementLevel`**, porque en aprendizaje no supervisado no se le dice al algoritmo qué grupos buscar; esa columna solo se usa al final para comparar.

K-Means y DBSCAN miden distancias, así que una variable con valores grandes (como el nivel) dominaría a una pequeña (como las sesiones). `StandardScaler` deja cada variable con media 0 y desviación 1. `InGamePurchases` es binaria (0 o 1) y se escala igual; su media equivale a la proporción de jugadores que compran.

In [ ]:
VARIABLES = ['PlayTimeHours', 'SessionsPerWeek', 'PlayerLevel', 'InGamePurchases']
faltan = [v for v in VARIABLES if v not in datos.columns]
if faltan:
    raise KeyError(f'Faltan columnas en el CSV: {faltan}. Columnas disponibles: {list(datos.columns)}')

X = datos[VARIABLES].dropna().copy()
print('Jugadores usados:', len(X), 'de', len(datos), '(se descartan filas con valores faltantes en estas variables)')
display(X.describe().T.round(2))

escalador = StandardScaler()
X_esc = escalador.fit_transform(X)
print('Media tras escalar (≈0):', X_esc.mean(axis=0).round(3))
print('Desviación tras escalar (≈1):', X_esc.std(axis=0).round(3))

MUESTRA_SIL = min(5000, len(X))   # la silueta compara todos los pares de puntos; se calcula con una muestra para que sea rápida

def silueta(matriz, etiquetas, excluir_ruido=False):
    etiquetas = np.asarray(etiquetas)
    if excluir_ruido:
        mascara = etiquetas != -1
        matriz, etiquetas = matriz[mascara], etiquetas[mascara]
    if len(set(etiquetas)) < 2:
        return np.nan
    try:
        return silhouette_score(matriz, etiquetas, sample_size=min(MUESTRA_SIL, len(etiquetas)), random_state=SEMILLA)
    except ValueError:
        return np.nan

## 4. Elegir el número de grupos: codo y silueta
- **Codo (inercia):** suma de distancias al cuadrado de cada jugador a su centroide. Siempre baja al aumentar *k*; interesa el punto donde la mejora deja de ser grande.
- **Coeficiente de silueta:** va de -1 a 1; mide qué tan cerca está cada jugador de su grupo comparado con el grupo vecino. Valores cercanos a 1 indican grupos bien separados; alrededor de 0, grupos que se traslapan.

In [ ]:
filas = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(X_esc)
    filas.append({'k': k, 'inercia': km.inertia_, 'silueta': silueta(X_esc, km.labels_)})
tabla_k = pd.DataFrame(filas)
display(tabla_k.round(4))

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].plot(tabla_k['k'], tabla_k['inercia'], marker='o')
ejes[0].set_title('Método del codo'); ejes[0].set_xlabel('Número de grupos (k)'); ejes[0].set_ylabel('Inercia')
ejes[1].plot(tabla_k['k'], tabla_k['silueta'], marker='o', color='tab:orange')
ejes[1].set_title('Coeficiente de silueta'); ejes[1].set_xlabel('Número de grupos (k)'); ejes[1].set_ylabel('Silueta')
for eje in ejes: eje.set_xticks(tabla_k['k'])
fig.tight_layout()
fig.savefig(FIGURAS / 'ns_01_codo_silueta.png', dpi=150)
plt.show()

### 4.1 Decisión de *k*
Se consideran valores de *k* entre 3 y 6: menos de 3 grupos no permite distinguir estilos (casual, intensivo, coleccionista) y más de 6 se vuelve difícil de interpretar. Dentro de ese rango se aplica un criterio de parsimonia: **se toma el *k* más pequeño cuya silueta esté a menos de 0.02 de la mejor**. Así, si la silueta es casi igual con 3 o con 6 grupos, se prefiere el modelo más simple. **El equipo puede fijar otro valor** en `K_MANUAL` si el codo o la interpretación lo justifican.

In [ ]:
K_MANUAL = None     # ejemplo: K_MANUAL = 4
candidatos_k = tabla_k[tabla_k['k'].between(3, 6)]
TOLERANCIA = 0.02
mejor_sil = candidatos_k['silueta'].max()
K = int(K_MANUAL) if K_MANUAL else int(candidatos_k[candidatos_k['silueta'] >= mejor_sil - TOLERANCIA]['k'].min())
print(f'Mejor silueta entre k=3 y 6: {mejor_sil:.3f}; se toma el menor k a menos de {TOLERANCIA} de ese valor.')
silueta_k = float(tabla_k.loc[tabla_k['k'] == K, 'silueta'].iloc[0])
print(f'k elegido: {K} | silueta: {silueta_k:.3f}')
if silueta_k >= 0.5:
    print('Lectura: estructura de grupos fuerte.')
elif silueta_k >= 0.25:
    print('Lectura: estructura razonable; los grupos se distinguen pero hay traslape.')
else:
    print('Lectura: estructura débil; los grupos se traslapan bastante y deben interpretarse como tendencias, no como categorías cerradas.')

## 5. K-Means final y estabilidad
El modelo final es un `Pipeline` (escalado + K-Means) para poder aplicarlo después a jugadores nuevos con los valores originales. Para comprobar que los grupos no dependen del azar, se repite el entrenamiento con otras semillas y se mide la coincidencia con el **índice de Rand ajustado** (1 = grupos idénticos, 0 = coincidencia como de azar).

In [ ]:
modelo_kmeans = Pipeline([
    ('escalador', StandardScaler()),
    ('kmeans', KMeans(n_clusters=K, n_init=10, random_state=SEMILLA)),
]).fit(X)
grupos = modelo_kmeans.predict(X)
X['grupo'] = grupos

aris = []
for semilla in range(100, 105):
    otro = KMeans(n_clusters=K, n_init=10, random_state=semilla).fit(X_esc)
    aris.append(adjusted_rand_score(grupos, otro.labels_))
print('Índice de Rand ajustado con 5 semillas distintas:', np.round(aris, 3))
print(f'Promedio: {np.mean(aris):.3f}', '(estable)' if np.mean(aris) >= 0.9 else '(los grupos cambian algo según la inicialización)')

## 6. Perfil de cada grupo
La tabla muestra los promedios en unidades originales. El mapa de calor muestra cuántas desviaciones estándar está cada grupo por encima (+) o por debajo (-) del promedio general; sirve para ver en qué se distingue cada grupo.

In [ ]:
medias = X.groupby('grupo')[VARIABLES].mean()
tamano = X['grupo'].value_counts().sort_index()
perfil = medias.round(2).copy()
perfil.insert(0, 'jugadores', tamano)
perfil.insert(1, 'porcentaje', (tamano / len(X) * 100).round(1))
display(perfil)

z = (medias - X[VARIABLES].mean()) / X[VARIABLES].std()
fig, ax = plt.subplots(figsize=(7, 0.9 * K + 1.5))
sns.heatmap(z, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, cbar_kws={'label': 'Desviaciones estándar vs. promedio'})
ax.set_title('Perfil de cada grupo (K-Means)'); ax.set_ylabel('Grupo'); ax.set_xlabel('')
fig.tight_layout()
fig.savefig(FIGURAS / 'ns_02_perfil_grupos.png', dpi=150)
plt.show()

## 7. Visualización con PCA en 2D
PCA resume las cuatro variables en dos componentes para poder dibujar a los jugadores. Se grafica una muestra aleatoria para que la figura sea legible. Las cargas indican qué variables pesan más en cada componente.

In [ ]:
pca = PCA(n_components=2, random_state=SEMILLA).fit(X_esc)
coords = pca.transform(X_esc)
print('Varianza explicada por PC1 y PC2:', (pca.explained_variance_ratio_ * 100).round(1), '%')
print('Varianza explicada acumulada:', round(pca.explained_variance_ratio_.sum() * 100, 1), '%')
display(pd.DataFrame(pca.components_.T, index=VARIABLES, columns=['PC1', 'PC2']).round(3))

idx = rng.choice(len(X), size=min(6000, len(X)), replace=False)
centroides_pca = pca.transform(modelo_kmeans.named_steps['kmeans'].cluster_centers_)
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(x=coords[idx, 0], y=coords[idx, 1], hue=grupos[idx], palette='tab10', s=14, alpha=0.6, ax=ax, legend='full')
ax.scatter(centroides_pca[:, 0], centroides_pca[:, 1], c='black', marker='X', s=160, label='Centroides')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f} %)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f} %)')
ax.set_title('Grupos K-Means proyectados con PCA'); ax.legend(title='Grupo', loc='best')
fig.tight_layout()
fig.savefig(FIGURAS / 'ns_03_pca_kmeans.png', dpi=150)
plt.show()

## 8. DBSCAN
DBSCAN no necesita fijar *k*: agrupa puntos densos y marca como **ruido** (`-1`) a los que quedan aislados. Depende de dos parámetros: `eps` (radio de vecindad) y `min_samples` (vecinos mínimos para formar un grupo denso).

Para elegir `eps` se usa la gráfica de distancia al vecino más cercano de orden `min_samples`: el "codo" de esa curva sugiere un radio razonable. Después se prueba una cuadrícula de valores y se compara número de grupos, porcentaje de ruido y silueta (calculada sin los puntos de ruido).

In [ ]:
MIN_BASE = 2 * len(VARIABLES)     # regla práctica: 2 x número de variables
distancias, _ = NearestNeighbors(n_neighbors=MIN_BASE).fit(X_esc).kneighbors(X_esc)
dist_k = np.sort(distancias[:, -1])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dist_k)
ax.set_title(f'Distancia al vecino {MIN_BASE} ordenada (para elegir eps)')
ax.set_xlabel('Jugadores ordenados'); ax.set_ylabel('Distancia')
fig.tight_layout()
fig.savefig(FIGURAS / 'ns_04_dbscan_distancias.png', dpi=150)
plt.show()

filas_db = []
for min_s in sorted({MIN_BASE, 2 * MIN_BASE}):
    for eps in [0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.2]:
        etq = DBSCAN(eps=eps, min_samples=min_s).fit_predict(X_esc)
        n_grupos = len(set(etq)) - (1 if -1 in etq else 0)
        filas_db.append({'eps': eps, 'min_samples': min_s, 'grupos': n_grupos,
                         'ruido_pct': round((etq == -1).mean() * 100, 2),
                         'silueta': silueta(X_esc, etq, excluir_ruido=True)})
tabla_db = pd.DataFrame(filas_db)
display(tabla_db.round(4))

### 8.1 Elección de parámetros de DBSCAN
Se prefieren configuraciones con al menos 2 grupos y máximo 20 % de ruido, y entre ellas la de mayor silueta. Se pueden fijar a mano con `EPS_MANUAL` y `MIN_MANUAL`.

In [ ]:
EPS_MANUAL, MIN_MANUAL = None, None   # ejemplo: EPS_MANUAL, MIN_MANUAL = 0.6, 8
if EPS_MANUAL and MIN_MANUAL:
    EPS, MIN_S = float(EPS_MANUAL), int(MIN_MANUAL)
else:
    validos = tabla_db[(tabla_db['grupos'] >= 2) & (tabla_db['ruido_pct'] <= 20)]
    if validos.empty:
        validos = tabla_db[tabla_db['grupos'] >= 1]
    if validos.empty:
        validos = tabla_db
    mejor = validos.sort_values(['silueta', 'ruido_pct'], ascending=[False, True], na_position='last').iloc[0]
    EPS, MIN_S = float(mejor['eps']), int(mejor['min_samples'])

etq_db = DBSCAN(eps=EPS, min_samples=MIN_S).fit_predict(X_esc)
n_grupos_db = len(set(etq_db)) - (1 if -1 in etq_db else 0)
ruido_db = (etq_db == -1).mean() * 100
silueta_db = silueta(X_esc, etq_db, excluir_ruido=True)
print(f'DBSCAN elegido: eps={EPS}, min_samples={MIN_S}')
print(f'Grupos: {n_grupos_db} | Ruido: {ruido_db:.2f} % | Silueta (sin ruido): {silueta_db:.3f}')
display(pd.Series(etq_db).value_counts().sort_index().rename('jugadores').to_frame())

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(x=coords[idx, 0], y=coords[idx, 1], hue=etq_db[idx], palette='tab10', s=14, alpha=0.6, ax=ax, legend='full')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title(f'DBSCAN (eps={EPS}, min_samples={MIN_S}) proyectado con PCA; -1 = ruido'); ax.legend(title='Grupo', loc='best')
fig.tight_layout()
fig.savefig(FIGURAS / 'ns_05_pca_dbscan.png', dpi=150)
plt.show()

## 9. Comparación de K-Means y DBSCAN

In [ ]:
comparacion = pd.DataFrame([
    {'algoritmo': 'K-Means', 'grupos': K, 'ruido_pct': 0.0, 'silueta': silueta_k},
    {'algoritmo': 'DBSCAN', 'grupos': n_grupos_db, 'ruido_pct': round(ruido_db, 2), 'silueta': silueta_db},
]).set_index('algoritmo')
display(comparacion.round(4))
if n_grupos_db >= 2 and ruido_db < 100:
    mascara = etq_db != -1
    print('Coincidencia entre ambos (índice de Rand ajustado, sin ruido):',
          round(adjusted_rand_score(grupos[mascara], etq_db[mascara]), 3))

## 10. Nombrar los estilos y proponer contenido
Los nombres se **sugieren** con una regla simple a partir del perfil: el grupo con menor ritmo de juego (horas, sesiones y nivel) es *Casual*, el de mayor ritmo es *Intensivo*, y entre los restantes el que más compra es *Coleccionista*; el resto se llama *Regular*. Es solo un punto de partida: **revisen la tabla y corrijan los nombres en `NOMBRES_MANUAL`** si el perfil real sugiere otra cosa.

In [ ]:
NOMBRES_MANUAL = {}     # ejemplo: {0: 'Casual', 1: 'Intensivo', 2: 'Coleccionista'}

def sugerir_nombres(tabla_z):
    ritmo = tabla_z[['PlayTimeHours', 'SessionsPerWeek', 'PlayerLevel']].mean(axis=1)
    orden = ritmo.sort_values().index.tolist()
    nombres = {orden[0]: 'Casual'}
    if len(orden) > 1:
        nombres[orden[-1]] = 'Intensivo'
    resto = [g for g in orden if g not in nombres]
    if resto:
        nombres[tabla_z.loc[resto, 'InGamePurchases'].idxmax()] = 'Coleccionista'
    contador = 0
    for g in orden:
        if g not in nombres:
            contador += 1
            nombres[g] = 'Regular' if contador == 1 else f'Regular {contador}'
    return nombres

nombres = {**sugerir_nombres(z), **{int(g): n for g, n in NOMBRES_MANUAL.items()}}

CONTENIDO = {
    'Casual': 'Misiones cortas de 5 a 10 minutos, recompensas diarias fáciles, tutoriales opcionales y recordatorios amables.',
    'Intensivo': 'Desafíos de nivel alto, modos competitivos con clasificación, eventos de temporada y dificultad escalable.',
    'Coleccionista': 'Tienda con cosméticos y paquetes, logros coleccionables, pase de temporada y ofertas personalizadas.',
    'Regular': 'Metas semanales progresivas, eventos que inviten a aumentar las sesiones y contenido de nivel medio.',
}
def contenido_para(nombre):
    base = nombre.split(' ')[0] if nombre.startswith('Regular') else nombre
    return CONTENIDO.get(base, 'Definir contenido según el perfil del grupo.')

estilos = perfil.copy()
estilos.insert(0, 'estilo', [nombres[g] for g in estilos.index])
estilos['contenido_propuesto'] = [contenido_para(nombres[g]) for g in estilos.index]
pd.set_option('display.max_colwidth', None)
display(estilos)

promedio = X[VARIABLES].mean()
print()
for g in estilos.index:
    f = estilos.loc[g]
    print(f"Grupo {g} · {f['estilo']}: {f['porcentaje']:.1f} % de los jugadores; "
          f"{f['PlayTimeHours']:.1f} h jugadas (promedio general {promedio['PlayTimeHours']:.1f}), "
          f"{f['SessionsPerWeek']:.1f} sesiones por semana ({promedio['SessionsPerWeek']:.1f}), "
          f"nivel {f['PlayerLevel']:.0f} ({promedio['PlayerLevel']:.0f}) y {f['InGamePurchases']*100:.0f} % realiza compras ({promedio['InGamePurchases']*100:.0f} %).")

## 11. Relación con el nivel de participación (comparación, no entrenamiento)
`EngagementLevel` no se usó para formar los grupos. Compararlo después muestra si los estilos descubiertos se relacionan con la participación que el aprendizaje supervisado intenta predecir.

In [ ]:
if 'EngagementLevel' in datos.columns:
    X['estilo'] = X['grupo'].map(nombres)
    nivel = datos.loc[X.index, 'EngagementLevel']
    tabla_cruce = (pd.crosstab(X['estilo'], nivel, normalize='index') * 100).round(1)
    columnas = [c for c in ['Low', 'Medium', 'High'] if c in tabla_cruce.columns]
    display(tabla_cruce[columnas + [c for c in tabla_cruce.columns if c not in columnas]])
    fig, ax = plt.subplots(figsize=(7, 4))
    tabla_cruce[columnas].plot.bar(stacked=True, ax=ax)
    ax.set_title('Nivel de participación dentro de cada estilo'); ax.set_xlabel('Estilo'); ax.set_ylabel('% de jugadores')
    ax.tick_params(axis='x', rotation=0); ax.legend(title='EngagementLevel', bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.tight_layout()
    fig.savefig(FIGURAS / 'ns_06_estilo_vs_participacion.png', dpi=150)
    plt.show()
else:
    print('El CSV no tiene la columna EngagementLevel; se omite esta comparación.')

## 12. Guardar el modelo
Se guarda el `Pipeline` (escalado + K-Means) junto con las variables, los nombres de los estilos y el PCA, todo en `models/kmeans_estilos.joblib`. Así la aplicación final puede clasificar a un jugador nuevo sin volver a entrenar. También se guardan las asignaciones en `data/processed/`.

In [ ]:
RUTA_MODELO = MODELOS / 'kmeans_estilos.joblib'
joblib.dump({
    'pipeline': modelo_kmeans,
    'pca': pca,
    'variables': VARIABLES,
    'k': K,
    'nombres': nombres,
    'contenido': {nombres[g]: contenido_para(nombres[g]) for g in nombres},
    'silueta': silueta_k,
    'semilla': SEMILLA,
}, RUTA_MODELO)
print('Modelo guardado en:', RUTA_MODELO.relative_to(RAIZ))

asignaciones = datos.loc[X.index].copy()
asignaciones['grupo'] = X['grupo']
asignaciones['estilo'] = X['grupo'].map(nombres)
asignaciones.to_csv(PROCESADOS / 'jugadores_con_estilo.csv', index=False)
print('Asignaciones guardadas en:', (PROCESADOS / 'jugadores_con_estilo.csv').relative_to(RAIZ))

# Comprobación: recargar el modelo y clasificar jugadores nuevos
def clasificar_jugador(horas, sesiones, nivel, compras, ruta=RUTA_MODELO):
    m = joblib.load(ruta)
    fila = pd.DataFrame([[horas, sesiones, nivel, compras]], columns=m['variables'])
    g = int(m['pipeline'].predict(fila)[0])
    return g, m['nombres'][g], m['contenido'][m['nombres'][g]]

for ejemplo in [(2, 3, 10, 0), (20, 15, 80, 0), (10, 8, 50, 1)]:
    print(ejemplo, '->', clasificar_jugador(*ejemplo)[:2])

## 13. Interpretación y acuerdos del equipo
Completar con los resultados reales de la ejecución.

1. **Elección de *k*:** ¿qué mostraron el codo y la silueta? ¿Qué *k* se eligió y por qué?
2. **Estabilidad:** ¿los grupos se repiten con otras semillas?
3. **Estilos:** ¿cómo se llama y cómo es cada grupo?
4. **DBSCAN:** ¿cuántos grupos y cuánto ruido encontró? ¿En qué coincide o difiere de K-Means?
5. **PCA:** ¿qué porcentaje de varianza explican PC1 y PC2? ¿Se ven separados los grupos?
6. **Contenido propuesto:** ¿qué cambio de diseño o contenido se probaría con cada estilo?
7. **Relación con participación:** ¿los estilos se asocian con Low, Medium y High?